In [0]:
df = spark.read.load("/Volumes/dai/phase2/bronze/customer_df_without_target")

display(df.head(10))

CustomerID,total_spent,total_transactions,total_quantity,last_purchase_date
17420,598.8300000000002,30,265,2011-10-20
16552,379.72999999999996,17,219,2011-04-11
17572,226.75,12,95,2011-09-29
15350,115.65,5,51,2010-12-01
12921,16389.740000000016,741,9454,2011-12-06
13090,8689.390000000003,161,2194,2011-12-01
14135,4690.31,134,3850,2011-12-08
12915,363.65,22,93,2011-07-14
17685,3191.5299999999993,130,1956,2011-11-23
17581,10736.109999999991,452,5861,2011-12-09


In [0]:
df.count()

4373

In [0]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- last_purchase_date: date (nullable = true)



## Create Initial Delta Table
---

In [0]:
# df.write.format("delta").saveAsTable("dai.phase2.customer_df_without_target")

## Create New Version By Appending New Records To The Data

---

In [0]:
# Append New Records
from datetime import date

data=[(1,2459.67,1,10,date(2026,3,2)),
      (2,1234.56,2,20,date(2026,3,3)),
      (3,3456.78,3,30,date(2026,3,4))]

new_df = spark.createDataFrame(data,schema=df.schema)
display(new_df)

new_df.write.format("delta").mode("append").saveAsTable("dai.phase2.customer_df_without_target")
display("## New Version Created")

CustomerID,total_spent,total_transactions,total_quantity,last_purchase_date
1,2459.67,1,10,2026-03-02
2,1234.56,2,20,2026-03-03
3,3456.78,3,30,2026-03-04


'## New Version Created'

#### Checking the Delta Table History for Versions:

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "dai.phase2.customer_df_without_target")
history_df = delta_table.history()
display(history_df)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-03-02T12:41:14.000Z,73331271719467,kalyanmistcse@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(4422701728491203),79d8e160-71d1-49c6-9e5a-bdd4d0969fb4,0302-120425-p9pvoba7-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1634)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-02T12:37:53.000Z,73331271719467,kalyanmistcse@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4422701728491203),b77a6e25-d77f-4e41-838d-86b884687de4,0302-120425-p9pvoba7-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4373, numOutputBytes -> 52328)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


### Intentionally saving the first row as the table data; simulating the problem of data loss in the data manipulation or feature engineering
---

In [0]:
new_df = new_df.limit(1)
display(new_df)

new_df.write.format("delta").mode("overwrite").saveAsTable("dai.phase2.customer_df_without_target")

CustomerID,total_spent,total_transactions,total_quantity,last_purchase_date
1,2459.67,1,10,2026-03-02


In [0]:
modified_df = spark.table('dai.phase2.customer_df_without_target')
display(modified_df)

CustomerID,total_spent,total_transactions,total_quantity,last_purchase_date
1,2459.67,1,10,2026-03-02


### Checking the version of this `modified_df`

In [0]:
display(history_df)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-02T12:50:27.000Z,73331271719467,kalyanmistcse@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4422701728491203),8af83fe0-4562-46bb-a0f9-e9d17b8fefd4,0302-120425-p9pvoba7-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 53962, numDeletionVectorsRemoved -> 0, numOutputRows -> 1, numOutputBytes -> 1571)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T12:41:14.000Z,73331271719467,kalyanmistcse@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(4422701728491203),79d8e160-71d1-49c6-9e5a-bdd4d0969fb4,0302-120425-p9pvoba7-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1634)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-02T12:37:53.000Z,73331271719467,kalyanmistcse@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4422701728491203),b77a6e25-d77f-4e41-838d-86b884687de4,0302-120425-p9pvoba7-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4373, numOutputBytes -> 52328)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


## Querying Older Version (Rollback):

---

In [0]:
df_old = spark.read.format("delta").option("versionAsOf", 0) \
    .table("dai.phase2.customer_df_without_target")

display(df_old.head(10))

CustomerID,total_spent,total_transactions,total_quantity,last_purchase_date
17420,598.8300000000002,30,265,2011-10-20
16552,379.72999999999996,17,219,2011-04-11
17572,226.75,12,95,2011-09-29
15350,115.65,5,51,2010-12-01
12921,16389.740000000016,741,9454,2011-12-06
13090,8689.390000000003,161,2194,2011-12-01
14135,4690.31,134,3850,2011-12-08
12915,363.65,22,93,2011-07-14
17685,3191.5299999999993,130,1956,2011-11-23
17581,10736.109999999991,452,5861,2011-12-09


## Comparing the difference between old and new versioned data:

---

In [0]:
diff_df = modified_df.join(df_old, on="CustomerID", how="left_anti")
display(diff_df)

CustomerID,total_spent,total_transactions,total_quantity,last_purchase_date
1,2459.67,1,10,2026-03-02
